In [1]:
from libraries.inference_training import Configuration, ImageDataset
from libraries.inference_training import initCudaEnvironment, createTransforms
from libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from libraries.inference_training import trainModel, saveModel, loadModel
import random

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [3]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    :param trainDirectory: path naar training dataset
    :param testDirectory: path naar testing dataset
    :param modelName: naam van model
    :param epochs: hoeveelheid epochs
    :param labels: list van labels, geef normaal ["parkeerplaatsen"] als er geen andere objecten zijn
    :param augment_data: bepaald of er image transforms gedaan worden, nog niet getest
    :param save_path: path naar save locatie van model
    :param maskdata: list van floats 
    :param save_interval: 
    :param model_description: beschrijft het model in de ONNX als het gesaved is
    :return:
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [ ]:
train_directory = "<insert train path here>"
test_directory = "<insert test path here>"
modelName = "<insert name here>"
epochs = 1
labels = ["<insert labels here>"]
augment = False
save_path = "<insert path to save location for model here>"
save_Interval = 0 # model is saved in between these amount of epochs
createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

# combo model with augment

In [4]:
train_directory = "C:/xxx/datasets/combo_overlay_sets/train"
test_directory = "C:/xxx/datasets/combo_overlay_sets/test"
modelName = "combo_model_with_augment"
epochs = 25
labels = ["parking_space"]  
augment = True
save_path = "C:/xxx/models/combo_models/augment/"
save_Interval = 5
model, config = createModel(train_directory, test_directory, modelName, epochs, labels, augment, save_path, save_Interval)

Device: cuda
Train Image count: 1600
Test Image count: 800
Pytorch model name C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/combo_model_with_augment.pt
Onnx file name C:/Users/Gebruiker/Desktop/homework/deep_learning_in_practice/models/combo_models/augment/combo_model_with_augment.onnx


C:\Users\Gebruiker\Desktop\homework\deep_learning_in_practice\libraries\engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/800]  eta: 0:48:17  lr: 0.000011  loss: 8.2996 (8.2996)  loss_classifier: 1.1845 (1.1845)  loss_box_reg: 0.1927 (0.1927)  loss_mask: 3.7019 (3.7019)  loss_objectness: 3.0555 (3.0555)  loss_rpn_box_reg: 0.1651 (0.1651)  time: 3.6222  data: 0.0647  max mem: 1661
Epoch: [0]  [ 10/800]  eta: 0:31:31  lr: 0.000074  loss: 5.5327 (6.1096)  loss_classifier: 1.1665 (1.1362)  loss_box_reg: 0.1026 (0.1019)  loss_mask: 3.5378 (3.4689)  loss_objectness: 0.9232 (1.2403)  loss_rpn_box_reg: 0.0977 (0.1623)  time: 2.3943  data: 0.0367  max mem: 1831
Epoch: [0]  [ 20/800]  eta: 0:30:30  lr: 0.000136  loss: 3.0648 (4.2548)  loss_classifier: 0.7518 (0.8438)  loss_box_reg: 0.0977 (0.1036)  loss_mask: 1.9323 (2.3403)  loss_objectness: 0.4908 (0.8520)  loss_rpn_box_reg: 0.0376 (0.1151)  time: 2.2828  data: 0.0371  max mem: 1831
Epoch: [0]  [ 30/800]  eta: 0:29:59  lr: 0.000199  loss: 1.6933 (3.3479)  loss_classifier: 0.3307 (0.6502)  loss_box_reg: 0.1107 (0.1184)  loss_mask: 0.7379 (1.7978) 

C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torch\nn\functional.py:4511: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  * torch.tensor(scale_factors[i], dtype=torch.float32)
C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torchvision\ops\boxes.py:166: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  boxes_x = torch.min(boxes_x, torch.tensor(width, dtype=boxes.dtype, device=boxes.device))
C:\Users\Gebruiker\miniconda3\envs\tygronai\Lib\site-packages\torchvision\ops\boxes.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sour

TypeError: sequence item 1: expected str instance, list found

^ AP and AR converge to the results at the 15th epoch and don't change after with the current dataset.